In [2]:
import cv2
import torch
import pandas as pd
import supervision as sv
from trackers import SORTTracker

# Detection

MegaDetector is an open-source AI model developed by Microsoft to automatically detect animals, humans, and vehicles in camera trap images. It is built directly on the YOLO object detection architecture, leveraging its real-time processing capabilities to filter out empty frames and speed up wildlife monitoring.

For tracking details see the following reference: https://trackers.roboflow.com/latest/trackers/sort/#run-on-video-webcam-or-rtsp-stream

In [3]:
# Load model.
model_path = "md_v5a.0.1.pt"
model_detection = torch.hub.load("ultralytics/yolov5", "custom", path=model_path, trust_repo=True)

Using cache found in /home/pietr/.cache/torch/hub/ultralytics_yolov5_master


YOLOv5 🚀 2026-9-5 Python-3.11.16 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)

Fusing layers... 
Model summary: 332 layers, 139,990,096 parameters, 0 gradients, 207.9 GFLOPs
Adding AutoShape... 


In [46]:
# Get input data.
video_path = "./videos/youtube/bear.mp4"
output_video_path = "output.mp4"

In [47]:
# Set confidence threshold for keeping trustable boxes.
confidence_threshold = 0.8

In [48]:
# Open input video.
video = cv2.VideoCapture(video_path)
# Get video properties.
fps = int(video.get(cv2.CAP_PROP_FPS))
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
# Create output video containing motion detection and related predictions.
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
output_video = cv2.VideoWriter(output_video_path, fourcc, fps=fps, frameSize=(width, height))

# Initialize tracker.
tracker = SORTTracker()
# Initialize visualization tools.
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

tracking_data = []
frame_id = 0
while video.isOpened():
    success, frame = video.read()
    
    if not success:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Detection.
    results = model_detection(frame_rgb)
    df_detections = results.pandas().xyxy[0]
    df_detections = df_detections[df_detections["confidence"] >= confidence_threshold]

    # Tracking.
    if not df_detections.empty:
        xyxy = df_detections[["xmin", "ymin", "xmax", "ymax"]].to_numpy()
        confidence = df_detections["confidence"].to_numpy()
        class_id = df_detections["class"].to_numpy()
        detections = sv.Detections(xyxy=xyxy, confidence=confidence, class_id=class_id)
    else:
        detections = sv.Detections.empty()
        
    detections = tracker.update(detections)

    # Store detection and tracking results.
    for xyxy_box, confidence, class_id, tracker_id in zip(
            detections.xyxy,
            detections.confidence,
            detections.class_id,
            detections.tracker_id,
        ):
            tracking_data.append(
                {
                    "frame_id": frame_id,
                    "tracker_id": tracker_id,
                    "xmin": xyxy_box[0],
                    "ymin": xyxy_box[1],
                    "xmax": xyxy_box[2],
                    "ymax": xyxy_box[3],
                    "confidence": confidence,
                    "class_id": class_id,
                }
            )

    # Visualization.
    labels = [f"#{tracker_id}" for tracker_id in detections.tracker_id]
    frame = box_annotator.annotate(scene=frame, detections=detections)
    frame = label_annotator.annotate(scene=frame, detections=detections, labels=labels)

    output_video.write(frame)

    frame_id += 1

video.release()
output_video.release()
cv2.destroyAllWindows()

In [49]:
df = pd.DataFrame(tracking_data)
df

,frame_id,tracker_id,xmin,ymin,xmax,ymax,confidence,class_id
0,0,-1,666.807434,211.015930,865.024231,428.872314,0.952980,0
1,1,-1,665.378723,207.579163,862.188171,428.663269,0.953875,0
2,2,0,663.518250,205.376907,859.107971,428.252258,0.952846,0
3,3,0,660.766541,203.023773,857.536316,427.769318,0.950315,0
4,4,0,657.344116,199.051270,855.066528,427.799194,0.945389,0
...,...,...,...,...,...,...,...,...
209,209,0,0.000000,10.271484,167.798950,645.739746,0.895911,0
210,210,0,0.000000,6.916962,157.894028,654.947632,0.875095,0
211,211,0,0.034523,4.583099,147.677246,666.479614,0.852541,0
212,212,0,0.000000,3.667236,136.363190,676.041382,0.826644,0


In [50]:
df["tracker_id"].unique()

array([-1,  0])